In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchdiffeq import odeint
from util.gaussian_process import GPPrior
from util.util import make_grid, reshape_for_batchwise, plot_loss_curve, plot_samples
import time

from util.util import load_navier_stokes
from torch.utils.data import TensorDataset, DataLoader

from models.fno import FNO
from models.FFM import FFMModel

from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

In [48]:
# All required constants
ntr = 20000
batch_size = 256

modes = 64
visch = 1
hch = 64
pch = 64
xdim = 1
t_scaling = 1000

device = 'cuda' if torch.cuda.is_available() else 'cpu'

lengthscale = 0.001
var = 1.0
sigma_min = 1e-4

lr = 1e-4

nepoch = 10
evalint = 5
saveint = 10
generate = True
spath = './models/'

In [58]:
!gdown https://drive.google.com/uc?id=1rdaAAQqsz8vePMLd8iETgdk-35Twsc72

Error:

	[WinError 3] The system cannot find the path specified: "'."

To report issues, please visit https://github.com/wkentaro/gdown/issues.


In [30]:
# Load the data
data = torch.from_numpy(np.load('./fno_ns_Re20_N5000_T50.npy'))

In [31]:
data.shape

torch.Size([50, 64, 64, 5000])

In [32]:
data = data.permute(3, 1, 2, 0)

In [33]:
data.shape

torch.Size([5000, 64, 64, 50])

In [34]:
# PreProcess the data

data = data.permute(0, -1, 1, 2).reshape(-1, 64, 64).unsqueeze(1) 
idx = torch.randperm(data.shape[0])
data = data[idx]

In [35]:
data.shape

torch.Size([250000, 1, 64, 64])

In [36]:
# Data loaders

data_tr, data_te = data[:ntr], data[ntr:]
loader_tr = DataLoader(data_tr, batch_size=batch_size, shuffle=True)
loader_te = DataLoader(data_te, batch_size=batch_size, shuffle=True)

In [51]:
# Create the model
model = FNO(modes, visch, hch, pch)
model.to(device)
model_wrapper = FFMModel(model, 
                         kernel_length=lengthscale, 
                         kernel_variance=var, 
                         sigma_min=sigma_min, 
                         device=device)

In [ ]:
optimizer = Adam(model.parameters(), lr)
scheduler = StepLR(optimizer, step_size=25, gamma=0.1)

In [53]:
model_wrapper.train(loader_tr,
                    optimizer,
                    epochs=nepoch,
                    scheduler=scheduler,
                    test_loader=loader_te,
                    eval_int=evalint,
                    save_int=saveint,
                    generate=generate,
                    save_path=spath
                   )

AssertionError: Error: inputs must have same number of dimensions

In [8]:
# Define the training loop 

In [9]:
# Model evaluation and inference

In [23]:
data.shape[2:]

torch.Size([64, 64])